In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import pandas as pd

#membaca file csv menggunakan pandas
df= pd.read_csv("/content/drive/MyDrive/Pratikum_Ml/Pratikum12/data/data.csv")


#cetak headet data (baris data) dari file
df.head()


In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df_clean = df.drop(['id', 'Unnamed: 32'], axis=1)

In [ ]:
df.duplicated().sum()

In [ ]:
features = df_clean.columns.drop('diagnosis')
X = df_clean[features].values

y = df_clean['diagnosis'].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Ukuran data X_train:", X_train.shape)
print("Ukuran data X_test", X_test.shape)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled[:5]

In [ ]:
svm_no_pca = SVC(kernel='rbf', gamma='scale', random_state=42)
svm_no_pca.fit(X_train_scaled, y_train)

y_pred_no_pca = svm_no_pca.predict(X_test_scaled)

acc_no_pca = accuracy_score(y_test, y_pred_no_pca)
print("Akurasi SVM tanpa PCA:", acc_no_pca)

print("\nClassification Report (tanpa PCA):")
print(classification_report(y_test, y_pred_no_pca, target_names=['Benign (B)', 'Malignant (M)']))

In [ ]:
pca = PCA(n_components=3)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("Shape X_train_pca: ", X_train_pca.shape)
print("Shape X_test_pca: ", X_test_pca.shape)

In [ ]:
explained_var = pca.explained_variance_ratio_
print("Explained Variance Ratio tiap komponen: ", explained_var)
print(" Total variansi yang dijelaskan 2 komponen pertama: ", explained_var.sum())

In [ ]:
plt.bar([1, 2, 3], explained_var)
plt.xlabel('Komponen Utama')
plt.ylabel('Varian yang dijelaskan')
plt.title('Variansi yg dijelaskan olh 3 komponen PCA')
plt.show()

In [ ]:
svm_pca = SVC(kernel='rbf', gamma='scale', random_state=42)
svm_pca.fit(X_train_pca, y_train)

y_pred_pca = svm_pca.predict(X_test_pca)

acc_pca = accuracy_score(y_test, y_pred_pca)
print("Akurasi SVM dengan PCA (3 komponen):", acc_pca)

print("\nClassification Report (dengan PCA):")
print(classification_report(y_test, y_pred_no_pca, target_names=['Benign (B)', 'Malignant (M)']))

In [ ]:
y_train_encoded = [0 if label == 'B' else 1 for label in y_train]

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

scatter = ax.scatter(
    X_train_pca[:, 0],
    X_train_pca[:, 1],
    X_train_pca[:, 2],
    c=y_train_encoded,
    cmap='viridis',
    s=60,
    alpha=0.7
)

ax.set_title('Visualisasi PCA (3 Komponen) - Dataset Breast Cancer')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')

legend1 = ax.legend(
    handles=scatter.legend_elements()[0],
    labels=['Benign (B)', 'Malignant (M)'],
    title="Diagnosis"
)
ax.add_artist(legend1)

plt.show()

In [ ]:
comparison = pd.DataFrame({
    'Model': ['SVM Tanpa PCA', 'SVM Dengan PCA (3 Komponen)'],
    'Jumlah Fitur': [X_train_scaled.shape[1], X_train_pca.shape[1]],
    'Akurasi': [acc_no_pca, acc_pca],
    'Variansi Total PCA': [None, explained_var.sum()]
})

# Tampilkan hasil
comparison

In [ ]:

plt.figure(figsize=(6,4))

plt.bar(['Tanpa PCA', 'Dengan PCA (3 Komponen)'], [acc_no_pca, acc_pca])

plt.title('Perbandingan Akurasi Model SVM')
plt.ylabel('Akurasi')
plt.ylim(0, 1)

for i, v in enumerate([acc_no_pca, acc_pca]):
    plt.text(i, v + 0.01, f'{v:.2f}', ha='center')

plt.show()